In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer,AutoModel,AutoModelForSeq2SeqLM
embedding_model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
generate_model_name="google/flan-t5-small"
embed_tokenizer=AutoTokenizer.from_pretrained(embedding_model_name)
embed_model=AutoModel.from_pretrained(embedding_model_name)
gen_tokenizer=AutoTokenizer.from_pretrained(generate_model_name)
gen_model=AutoModelForSeq2SeqLM.from_pretrained(generate_model_name)
documents=[
    "For fat loss dinner,choose high protien,low fat,and moderate carbs",
    "Good chest exercises include bench press,incline dumbbell press,and cable fly",
    "  Squats train the quadriceps,glutes,and core stability",
    "To gain high muscle,you need enough protien,calorie surplus,and progressive overload"
]
def encode_texts(texts):
    inputs=embed_tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs=embed_model(**inputs)
    token_embeddings=outputs.last_hidden_state
    attention_mask=inputs["attention_mask"]

    mask=attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    masked_embeddings=token_embeddings*mask

    sum_embeddings=torch.sum(masked_embeddings,dim=1)
    sum_mask=torch.clamp(mask.sum(dim=1),min=1e-9)

    sentence_embeddings=sum_embeddings/sum_mask
    sentence_embeddings=F.normalize(sentence_embeddings,p=2,dim=1)

    return sentence_embeddings
doc_embeddings=encode_texts(documents)
def retrieve_doc(query,documents,doc_embeddings,top_k=2):
    query_embedding=encode_texts([query])
    scores=torch.matmul(query_embedding,doc_embeddings.T)
    topk_values,topk_indices=torch.topk(scores,k=top_k,dim=1)
    retrieved_doc=[]
    for i in range(top_k):
        idx=topk_indices[0][i].item()
        score=topk_values[0][i].item()
        retrieved_doc.append({
            "doc_index":idx,
            "document":documents[idx],
            "score":score
        })
    return retrieved_doc
def build_prompt(query,retrieved_doc):
    context="\n".join(  
        [f"-{item['document']}" for item in retrieved_doc]
    )
    prompt=(
        "Answer the question using the context.\n"
        "If the context is relevant,use it clearly and briefly.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:{query}\n"
        "Answer:"
    )
    return prompt
def generate_answer(prompt):
    inputs=gen_tokenizer(prompt,return_tensors="pt",truncation=True)
    with torch.no_grad():
        output_ids=gen_model.generate(
            **inputs,
            max_new_tokens=64
        )
    answer=gen_tokenizer.decode(output_ids[0],skip_special_tokens=True)
    return answer
def rag_pipeline(query,documents,doc_embeddings,top_k=2):
    retrieved_doc=retrieve_doc(query,documents,doc_embeddings,top_k=top_k)
    prompt=build_prompt(query,retrieved_doc)
    answer=generate_answer(prompt)
    return{
        "query":query,
        "retrieved_doc":retrieved_doc,
        "prompt":prompt,
        "answer":answer
    }
def evaluate_retrieval(retrieved_doc,expected_doc_index):
    hit_at_1=1 if retrieved_doc[0]["doc_index"] == expected_doc_index else 0
    hit_at_2=1 if any(item["doc_index"] == expected_doc_index for item in retrieved_doc[:2]) else 0
    return hit_at_1,hit_at_2
def evaluate_answer(answer,expected_keywards):
    answer_lower=answer.lower()
    matched=0
    for keyward in expected_keywards:
        if keyward.lower() in answer_lower:
            matched += 1
    keyward_score=matched/len(expected_keywards)
    return keyward_score,matched
eval_samples=[
    {
        "query":"What should I eat for dinner during fat loss?",
        "expected_doc_index":0,
        "expected_keywards":["high protein","low fat","moderate carbs"]
    },
    {
        "query":"What ecercises should I do for chest day?",
        "expected_doc_index":1,
        "expected_keywards":["bench press","incline dumbbell press"]
    },
    {
        "query":"How can I train legs and glutes?",
        "expected_doc_index":2,
        "expected_keywards":["quadriceps","glutes"]
    },
    {
        "query":"How do I eat during muscle gain?",
        "expected_doc_index":3,
        "expected_keywards":["protein","calorie surplus"]
    }

]
total_hit1=0
total_hit2=0
total_keyward_score=0
for i,sample in enumerate(eval_samples):
    result=rag_pipeline(sample["query"],documents,doc_embeddings,top_k=2)

    hit1,hit2=evaluate_retrieval(
        result["retrieved_doc"],
        sample["expected_doc_index"]
    )
    keyward_score,matched=evaluate_answer(
        result["answer"],
        sample["expected_keywards"]
    )
    total_hit1 += hit1
    total_hit2 += hit2
    total_keyward_score += keyward_score

    print(f"\n===== Sample {i+1} =====")
    print("Query",sample["query"])
    print("Expected doc index",sample["expected_doc_index"])
    print("Retrieved top 2:")
    for rank,item in enumerate(result["retrieved_doc"]):
        print((f"  Rank {rank+1}:doc_index={item['doc_index']},score={round(item['score'],4)}")) 
        print("   ",item["document"])

    print("Generate answer",result["answer"])
    print("Hit@1:",hit1)
    print("Hit@2",hit2)
    print("Keyward matched",f"{matched}/{len(sample['expected_keywards'])}")
    print("Keyward score:",round(keyward_score,4))
avg_hit1=total_hit1/len(eval_samples)
avg_hit2=total_hit2/len(eval_samples)
avg_keyward_score=total_keyward_score/len(eval_samples)
print("\n===== Final Evaluation Summary =====")
print("Average Hit@1:",round(avg_hit1,4))
print("Average Hit@2:",round(avg_hit2,4))
print("Average Keyward Score:",round(avg_keyward_score,4))

c:\Users\marce\ai-career-2026\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5387.83it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4871.73it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence th


===== Sample 1 =====
Query What should I eat for dinner during fat loss?
Expected doc index 0
Retrieved top 2:
  Rank 1:doc_index=0,score=0.8199
    For fat loss dinner,choose high protien,low fat,and moderate carbs
  Rank 2:doc_index=3,score=0.2736
    To gain high muscle,you need enough protien,calorie surplus,and progressive overload
Generate answer high protien,low fat,and moderate carbs
Hit@1: 1
Hit@2 1
Keyward matched 2/3
Keyward score: 0.6667

===== Sample 2 =====
Query What ecercises should I do for chest day?
Expected doc index 1
Retrieved top 2:
  Rank 1:doc_index=1,score=0.6507
    Good chest exercises include bench press,incline dumbbell press,and cable fly
  Rank 2:doc_index=3,score=0.3506
    To gain high muscle,you need enough protien,calorie surplus,and progressive overload
Generate answer Good chest exercises include bench press,incline dumbbell press,and cable fly
Hit@1: 1
Hit@2 1
Keyward matched 2/2
Keyward score: 1.0

===== Sample 3 =====
Query How can I train legs